# Setup: dados sintéticos de vendas

Gera a tabela Delta `sales_summary` no lakehouse padrão (`lh_demo_google`) com dados **100% sintéticos**
(seed fixa, resultado reproduzível). Rode uma vez antes da demo.

> Anexe o lakehouse `lh_demo_google` como **lakehouse padrão** deste notebook antes de executar.

In [ ]:
import random
from datetime import date
from decimal import Decimal

import notebookutils  # provided by the Fabric runtime
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

# The /lakehouse/default FUSE mount does not support the file rename that a Delta commit needs
# ("Unable to rename file"), so writes go straight to OneLake through the abfss endpoint.
_context = notebookutils.runtime.context
WORKSPACE_ID = _context["defaultLakehouseWorkspaceId"]
LAKEHOUSE_ID = _context["defaultLakehouseId"]
TABLE_PATH = (
    f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/dbo/sales_summary"
)
STORAGE_OPTIONS = {
    "bearer_token": notebookutils.credentials.getToken("storage"),
    "use_fabric_endpoint": "true",
}
SEED = 42

REGIONS = ["North", "Northeast", "Midwest", "Southeast", "South"]
CATEGORIES = ["Accessories", "Bikes", "Clothing", "Components"]
MONTHS = [date(2026, month, 1) for month in range(1, 11)]  # 10 x 5 x 4 = 200 rows

SCHEMA = pa.schema(
    [
        ("order_month", pa.date32()),
        ("region", pa.string()),
        ("product_category", pa.string()),
        ("order_count", pa.int32()),
        ("revenue", pa.decimal128(18, 2)),
    ]
)

In [ ]:
def build_sample_rows(seed: int) -> list[dict[str, object]]:
    """Return one synthetic row per month x region x category."""
    rng = random.Random(seed)
    rows: list[dict[str, object]] = []
    for month in MONTHS:
        for region in REGIONS:
            for category in CATEGORIES:
                order_count = rng.randint(20, 400)
                avg_ticket = rng.uniform(35.0, 900.0)
                rows.append(
                    {
                        "order_month": month,
                        "region": region,
                        "product_category": category,
                        "order_count": order_count,
                        "revenue": Decimal(f"{order_count * avg_ticket:.2f}"),
                    }
                )
    return rows


table = pa.Table.from_pylist(build_sample_rows(SEED), schema=SCHEMA)
write_deltalake(TABLE_PATH, table, mode="overwrite", storage_options=STORAGE_OPTIONS)
print(f"Wrote {table.num_rows} rows to dbo.sales_summary")

In [ ]:
# Read back to validate the write.
df = DeltaTable(TABLE_PATH, storage_options=STORAGE_OPTIONS).to_pandas()
print(f"Rows in sales_summary: {len(df)}")
df.head()